# 43b. RG3 최종 후보 — 가스 RF(기존) + 미성형 N1(unlabeled 정상 보강) 결합

| 역할 | 모델 | 근거 |
|---|---|---|
| 가스 | RF, 원값 21 + 새 변수 3개의 직전 10샷 std (labeled만) | 37번 반복 19 (8/17, 순열 p 0.007), 41번 M0 |
| 미성형 | 같은 변수 + **unlabeled 3,000샷을 정상으로 추가(가중치 0.3)** | 41번 N1 (시드 3개 모두 3/8, AP 3배) |
| 결합 | 검사 샷 K 중 g×K는 가스 모델 상위, 나머지는 미성형 모델 순위 | g = 0.68(불량 수 비율 17:8) / 0.5, 미리 정함 |

- 평가: Run 하나씩 빼고 검증(LORO), 시드 3개 평균·범위
- 검사 비율 0.5~30%별 불량·가스·미성형 개수, Precision, Recall, F1, 무작위 대비 배수
- 마지막 셀: 순열검정(재학습 포함)으로 우연 여부 확인 → 보고서 숫자로 쓰려면 꼭 돌리기

**준비**: 20a·20b 실행. 실행 시간: 본 실험 수 분, 순열검정 n=50이면 수십 분

In [ ]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
import rg3_combo as RC
R = RC.run(seeds=(0, 1, 2))

## 1. 검사 10%에서 비교 (기준: 기존 가스 모델만 = 가스 7~8/17, 미성형 1~2/8)

In [ ]:
display(RC.at_rate(R, '10%').round(3))

## 2. 다른 검사 비율 (5%, 20%)

In [ ]:
display(RC.at_rate(R, '5%').round(3))
display(RC.at_rate(R, '20%').round(3))

## 3. 전체 표 (모델 × 검사 비율, 시드 평균)

In [ ]:
display(R['summary'].round(3))

## 4. 모델별 AP (시드별)

In [ ]:
display(R['ap'].round(3))

## 5. 검사 비율별 곡선

In [ ]:
RC.plot(R, seed=0, save='./outputs/rg3_combo/curves.png')

## 6. 검사 비율과 무관한 분류 지표 (시드별)
R-Precision, Precision@Recall 25·50·75%, 필요 검사 비율, ROC-AUC, AP — 최대 1, 검사량을 정하지 않음

In [ ]:
K = RC.rank_table(R)
display(K.round(3))
display(K.groupby('모델').mean(numeric_only=True).round(3))
K.to_csv('./outputs/rg3_combo/rank_metrics.csv', index=False, encoding='utf-8-sig')

## 7. 순열검정 (재학습 포함) — 보고서 숫자로 쓰려면 실행
라벨 위치를 Run 안에서 무작위로 옮기고 두 모델 학습·결합을 처음부터 반복 → 관측 포착 수가 우연을 넘는지 (p < 0.05)

In [ ]:
P = RC.permutation_test(n=50, gas_share=0.68, rate=0.10)
display(P)